#  Obstacle Detection for Autonomous Driving
## Using Semantic, Instance, and Transformer Models on the Waymo Open Dataset
### With Weather Robustness

---
**Dataset:** Waymo Open Dataset  
**Framework:** PyTorch  


---

### Architecture Overview
```
RGB Camera ──► ResNet50 / Swin-T Encoder ──┐
                                            ├──► BEVFusion Module ──► DETR3D Transformer ──► Output
LiDAR Point Cloud ──► PointPillars ─────────┘         │
                                                 Weather Module
                                                 Seg Head / GRU
```

### Outputs
- 3D Bounding Boxes (class + confidence)
- Semantic Segmentation Masks
- Bird's Eye View (BEV) Maps
- Trajectory Waypoints
- Weather Condition Score


## Cell 1 — Environment Setup & Dependency Installation

In [2]:

# CELL 1: Environment Setup
# Install all required packages for the pipeline


import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

packages = [
    'torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118',
    'albumentations',
    'opencv-python-headless',
    'matplotlib',
    'numpy',
    'scipy',
    'tqdm',
    'timm',              # Swin Transformer backbone
    'einops',            # Tensor rearrangement (Transformer ops)
    'open3d',            # LiDAR / point cloud visualization
    'waymo-open-dataset-tf-2-11-0',  # Official Waymo SDK (TF-based reader)
    'tensorflow==2.11.0',
    'pycocotools',       # COCO-style mAP evaluation
    'tensorboard',
]

print('Installing dependencies — this may take a few minutes...')
for pkg in packages:
    try:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split(),
            stderr=subprocess.DEVNULL
        )
        print(f'  ✓ {pkg.split()[0]}')
    except Exception as e:
        print(f'  ✗ {pkg.split()[0]} — {e}')

print('\nAll dependencies processed.')

Installing dependencies — this may take a few minutes...
  ✓ torch
  ✓ albumentations
  ✓ opencv-python-headless
  ✓ matplotlib
  ✓ numpy
  ✓ scipy
  ✓ tqdm
  ✓ timm
  ✓ einops
  ✓ open3d
  ✗ waymo-open-dataset-tf-2-11-0 — Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'waymo-open-dataset-tf-2-11-0']' returned non-zero exit status 1.
  ✗ tensorflow==2.11.0 — Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'tensorflow==2.11.0']' returned non-zero exit status 1.
  ✓ pycocotools
  ✓ tensorboard

All dependencies processed.


## Cell 2 — Imports & Reproducibility Seeds

In [3]:

# CELL 2: Global Imports & Reproducibility


import os, random, math, time, json, warnings
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')   # headless rendering in Colab
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import torchvision
from torchvision import transforms

import timm                      # Swin Transformer
import albumentations as A
from albumentations.pytorch import ToTensorV2
from einops import rearrange, repeat

from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Project directories ───────────────────────────────────────
ROOT = Path('/content/waymo_detection')
for d in ['data/raw', 'data/processed', 'checkpoints', 'logs', 'outputs/viz']:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

print(f'Project root: {ROOT}')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
Project root: /content/waymo_detection


## Cell 3 — Global Configuration

In [4]:

# CELL 3: Global Configuration Dataclass
# Central config object — change here, affects everything


from dataclasses import dataclass, field

@dataclass
class Config:
    # ── Data ────────────────────────────────────────────────
    img_h: int = 384          # input image height
    img_w: int = 640          # input image width
    num_classes: int = 10     # Waymo object classes
    num_seg_classes: int = 8  # road / lane segmentation
    max_lidar_pts: int = 20000
    voxel_size: Tuple = (0.2, 0.2, 4.0)   # (x,y,z) metres
    point_cloud_range: List = field(default_factory=lambda: [-50, -50, -3, 50, 50, 1])

    # ── Model ────────────────────────────────────────────────
    backbone: str = 'swin_small_patch4_window7_224'  # timm model name
    img_feat_dim: int = 768   # Swin-S output channels
    lidar_feat_dim: int = 256 # PointPillars output channels
    bev_h: int = 100          # BEV grid height
    bev_w: int = 100          # BEV grid width
    bev_feat_dim: int = 256
    num_queries: int = 300    # DETR object queries
    num_transformer_layers: int = 6
    num_heads: int = 8
    ffn_dim: int = 1024
    gru_hidden: int = 256     # temporal GRU hidden size

    # ── Training ─────────────────────────────────────────────
    batch_size: int = 4
    num_epochs: int = 50
    lr: float = 1e-4
    weight_decay: float = 1e-4
    grad_accum_steps: int = 4
    mixed_precision: bool = True
    early_stop_patience: int = 10
    lr_warmup_epochs: int = 5
    num_workers: int = 2

    # ── Weather augmentation probabilities ───────────────────
    fog_prob: float = 0.25
    rain_prob: float = 0.25
    snow_prob: float = 0.15
    night_prob: float = 0.20
    dust_prob: float = 0.15

    # ── Loss weights ─────────────────────────────────────────
    loss_cls_w: float = 1.0
    loss_box_w: float = 5.0
    loss_giou_w: float = 2.0
    loss_seg_w: float = 1.5
    loss_traj_w: float = 0.5

    # ── Misc ─────────────────────────────────────────────────
    class_names: List[str] = field(default_factory=lambda: [
        'Vehicle', 'Pedestrian', 'Cyclist', 'Sign', 'Truck',
        'Bus', 'Motorcycle', 'Trailer', 'Bicycle', 'Unknown'
    ])
    seg_class_names: List[str] = field(default_factory=lambda: [
        'Road', 'Lane', 'Sidewalk', 'Building',
        'Vegetation', 'Sky', 'Obstacle', 'Unknown'
    ])

CFG = Config()
print('Configuration loaded:')
print(f'  Image size   : {CFG.img_h} x {CFG.img_w}')
print(f'  Backbone     : {CFG.backbone}')
print(f'  Queries      : {CFG.num_queries}')
print(f'  Classes      : {CFG.class_names}')
print(f'  Batch size   : {CFG.batch_size}')
print(f'  Mixed prec.  : {CFG.mixed_precision}')

Configuration loaded:
  Image size   : 384 x 640
  Backbone     : swin_small_patch4_window7_224
  Queries      : 300
  Classes      : ['Vehicle', 'Pedestrian', 'Cyclist', 'Sign', 'Truck', 'Bus', 'Motorcycle', 'Trailer', 'Bicycle', 'Unknown']
  Batch size   : 4
  Mixed prec.  : True


## Cell 4 — Weather Augmentation Module

In [5]:

# CELL 4: Synthetic Weather Augmentation
# Physically-motivated simulations for:
#   fog, rain, snow, night, dust, motion blur
# Applied BOTH at dataset-load time AND as a domain adaptation
# regulariser during training.


class WeatherAugmentor:
    """Generates synthetic adverse weather conditions on RGB images."""

    # ── Fog ──────────────────────────────────────────────────
    @staticmethod
    def add_fog(img: np.ndarray, intensity: float = 0.5) -> np.ndarray:
        """
        Atmospheric scattering model (Koschmieder's law):
            I(x) = I0(x) * exp(-beta*d) + A*(1 - exp(-beta*d))
        Approximated with a depth-dependent haze overlay.
        """
        h, w = img.shape[:2]
        # Simulate depth gradient — objects further away fog more
        depth_map = np.tile(np.linspace(0.2, 1.0, h)[:, None], (1, w))
        fog_density = intensity * depth_map
        fog_layer = (np.ones_like(img) * 255 * fog_density[..., None]).astype(np.uint8)
        alpha = np.clip(fog_density * 0.8, 0, 0.9)[..., None]
        return np.clip(img * (1 - alpha) + fog_layer * alpha, 0, 255).astype(np.uint8)

    # ── Rain ─────────────────────────────────────────────────
    @staticmethod
    def add_rain(img: np.ndarray, intensity: int = 200,
                 streak_len: int = 20, angle: int = -10) -> np.ndarray:
        """Random rain streak simulation with motion blur."""
        h, w = img.shape[:2]
        rain_layer = np.zeros((h, w), dtype=np.uint8)

        # Draw rain streaks
        for _ in range(intensity):
            x1 = random.randint(0, w - 1)
            y1 = random.randint(0, h - 1)
            dx = int(streak_len * math.sin(math.radians(angle)))
            dy = int(streak_len * math.cos(math.radians(angle)))
            x2 = np.clip(x1 + dx, 0, w - 1)
            y2 = np.clip(y1 + dy, 0, h - 1)
            brightness = random.randint(180, 255)
            cv2.line(rain_layer, (x1, y1), (x2, y2), brightness, 1)

        # Slight motion blur along streak direction
        kernel_size = max(3, streak_len // 4)
        kernel = np.zeros((kernel_size, kernel_size))
        kernel[kernel_size // 2, :] = 1.0 / kernel_size
        M = cv2.getRotationMatrix2D((kernel_size//2, kernel_size//2), angle, 1)
        kernel = cv2.warpAffine(kernel, M, (kernel_size, kernel_size))
        rain_blur = cv2.filter2D(rain_layer, -1, kernel)

        # Blend with original
        rain_rgb = np.stack([rain_blur] * 3, axis=-1)
        result = np.clip(img.astype(np.float32) * 0.8 + rain_rgb * 0.2, 0, 255)
        return result.astype(np.uint8)

    # ── Snow ─────────────────────────────────────────────────
    @staticmethod
    def add_snow(img: np.ndarray, intensity: int = 400,
                 flake_size: int = 3) -> np.ndarray:
        """Gaussian snowflake simulation with brightness overlay."""
        h, w = img.shape[:2]
        snow_layer = np.zeros((h, w), dtype=np.float32)

        for _ in range(intensity):
            cx = random.randint(0, w - 1)
            cy = random.randint(0, h - 1)
            r = random.randint(1, flake_size)
            brightness = random.uniform(0.7, 1.0)
            y_lo, y_hi = max(0, cy-r), min(h, cy+r+1)
            x_lo, x_hi = max(0, cx-r), min(w, cx+r+1)
            snow_layer[y_lo:y_hi, x_lo:x_hi] = brightness

        # Slight Gaussian blur to soften flakes
        snow_blur = cv2.GaussianBlur(snow_layer, (5, 5), 0)
        snow_rgb = np.stack([snow_blur * 255] * 3, axis=-1)

        # Brightness boost (winter overcast sky)
        brightened = np.clip(img.astype(np.float32) * 1.1 + snow_rgb * 0.3, 0, 255)
        return brightened.astype(np.uint8)

    # ── Night / Low light ────────────────────────────────────
    @staticmethod
    def add_night(img: np.ndarray,
                  gamma: float = 2.5,
                  headlight_prob: float = 0.5) -> np.ndarray:
        """
        Gamma darkening + optional synthetic headlight glare.
        """
        # Gamma correction for darkness
        inv_gamma = 1.0 / gamma
        table = np.array([(i / 255.0) ** inv_gamma * 255
                          for i in range(256)], dtype=np.uint8)
        dark = cv2.LUT(img, table)

        # Add headlight glare blobs
        if random.random() < headlight_prob:
            h, w = dark.shape[:2]
            glare = dark.copy().astype(np.float32)
            for _ in range(random.randint(2, 4)):
                cx = random.randint(w // 4, 3 * w // 4)
                cy = random.randint(h // 2, h - 1)
                radius = random.randint(30, 80)
                overlay = np.zeros_like(glare)
                cv2.circle(overlay, (cx, cy), radius, (255, 255, 200), -1)
                overlay = cv2.GaussianBlur(overlay, (radius | 1, radius | 1), 0)
                glare = np.clip(glare + overlay * 0.6, 0, 255)
            return glare.astype(np.uint8)
        return dark

    # ── Dust / Sand ──────────────────────────────────────────
    @staticmethod
    def add_dust(img: np.ndarray, intensity: float = 0.4) -> np.ndarray:
        """Yellow-brown tinted haze to simulate dust/sand storms."""
        h, w = img.shape[:2]
        dust_color = np.array([180, 150, 80], dtype=np.float32)  # warm sand tone
        dust_layer = np.ones((h, w, 3), dtype=np.float32) * dust_color
        alpha = intensity * np.random.uniform(0.6, 1.0)
        return np.clip(
            img.astype(np.float32) * (1 - alpha) + dust_layer * alpha, 0, 255
        ).astype(np.uint8)

    # ── Motion blur ──────────────────────────────────────────
    @staticmethod
    def add_motion_blur(img: np.ndarray,
                        kernel_size: int = 15,
                        angle: float = 0.0) -> np.ndarray:
        """Directional motion blur mimicking camera shake."""
        k = np.zeros((kernel_size, kernel_size))
        k[kernel_size // 2, :] = 1.0 / kernel_size
        M = cv2.getRotationMatrix2D(
            (kernel_size // 2, kernel_size // 2), angle, 1
        )
        k = cv2.warpAffine(k, M, (kernel_size, kernel_size))
        return cv2.filter2D(img, -1, k)

    # ── Combined pipeline ─────────────────────────────────────
    def __call__(self, img: np.ndarray,
                 cfg: Config) -> Tuple[np.ndarray, str]:
        """
        Randomly select and apply ONE weather condition.
        Returns augmented image + condition label.
        """
        r = random.random()
        cum = 0.0
        probs = [
            (cfg.fog_prob,   'fog',   lambda x: self.add_fog(x,  random.uniform(0.3, 0.7))),
            (cfg.rain_prob,  'rain',  lambda x: self.add_rain(x, random.randint(100, 300))),
            (cfg.snow_prob,  'snow',  lambda x: self.add_snow(x, random.randint(200, 600))),
            (cfg.night_prob, 'night', lambda x: self.add_night(x, random.uniform(2.0, 3.5))),
            (cfg.dust_prob,  'dust',  lambda x: self.add_dust(x, random.uniform(0.3, 0.6))),
        ]
        for prob, name, fn in probs:
            cum += prob
            if r < cum:
                return fn(img), name
        return img, 'clear'


# ── LiDAR noise augmentation ──────────────────────────────────
def augment_lidar_weather(pts: np.ndarray, condition: str) -> np.ndarray:
    """
    Add sensor noise to LiDAR point cloud based on weather.
    - Fog/snow: random dropouts + range reduction
    - Rain:     intensity noise
    - Night:    Gaussian coordinate jitter
    """
    pts = pts.copy()

    if condition in ('fog', 'snow', 'dust'):
        # Drop 10-40% of points (scattering)
        keep = np.random.rand(len(pts)) > np.random.uniform(0.1, 0.4)
        pts = pts[keep]
        # Reduce effective range
        if len(pts) > 0:
            max_range = np.random.uniform(30, 60)
            ranges = np.linalg.norm(pts[:, :3], axis=1)
            pts = pts[ranges < max_range]

    elif condition == 'rain':
        # Add random false returns (water droplets)
        n_false = int(len(pts) * 0.05)
        false_pts = np.random.randn(n_false, pts.shape[1]) * 2.0
        pts = np.vstack([pts, false_pts])
        # Intensity noise
        if pts.shape[1] > 3:
            pts[:, 3] += np.random.randn(len(pts)) * 0.05

    elif condition == 'night':
        # Gaussian coordinate jitter (sensor thermal noise)
        pts[:, :3] += np.random.randn(*pts[:, :3].shape) * 0.02

    return pts


WEATHER_AUG = WeatherAugmentor()
print('WeatherAugmentor and LiDAR noise module ready.')

WeatherAugmentor and LiDAR noise module ready.


## Cell 5 — Dataset & DataLoader

In [6]:

# CELL 5: Waymo-Compatible Dataset
#
# Two modes:
#   1. REAL: reads Waymo .tfrecord files via the official SDK
#   2. SYNTHETIC: generates plausible random tensors for
#      development / Colab demo without the full dataset.
#
# Toggle with USE_SYNTHETIC = True/False


USE_SYNTHETIC = True   # Set False when Waymo TFRecords are available


# ── Albumentations pipeline ───────────────────────────────────
def build_transforms(cfg: Config, train: bool = True):
    """Standard photometric + geometric augmentation pipeline."""
    base = [
        A.Resize(cfg.img_h, cfg.img_w),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]
    if not train:
        return A.Compose(base, bbox_params=A.BboxParams(
            format='pascal_voc', label_fields=['class_labels']))

    aug = [
        A.Resize(cfg.img_h, cfg.img_w),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.3, contrast=0.3,
                      saturation=0.2, hue=0.1, p=0.5),
        A.GaussNoise(var_limit=(10, 50), p=0.3),
        A.GaussianBlur(blur_limit=(3, 7), p=0.2),
        A.RandomBrightnessContrast(p=0.4),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]
    return A.Compose(aug, bbox_params=A.BboxParams(
        format='pascal_voc', label_fields=['class_labels']))


class WaymoSyntheticDataset(Dataset):
    """
    Synthetic proxy dataset that mimics Waymo data structure.
    Generates realistic random tensors for all modalities.
    Replace with WaymoRealDataset (below) when TFRecords are
    available.
    """

    def __init__(self, cfg: Config, split: str = 'train',
                 size: int = 500):
        self.cfg = cfg
        self.split = split
        self.size = size
        self.train = (split == 'train')
        self.transforms = build_transforms(cfg, self.train)

    def __len__(self):
        return self.size

    def _make_image(self):
        """Synthetic RGB frame (H x W x 3, uint8)."""
        # Sky gradient + road gradient for realism
        h, w = self.cfg.img_h, self.cfg.img_w
        img = np.zeros((h, w, 3), dtype=np.uint8)
        for row in range(h):
            t = row / h
            # Sky (blue) → Road (grey)
            sky_col = np.array([100, 150, 210]) * (1 - t)
            road_col = np.array([80, 80, 80]) * t
            img[row] = np.clip(sky_col + road_col, 0, 255).astype(np.uint8)
        # Add random boxes to simulate objects
        for _ in range(random.randint(2, 8)):
            x = random.randint(0, w - 80)
            y = random.randint(h // 2, h - 40)
            bw = random.randint(40, 120)
            bh = random.randint(30, 80)
            color = [random.randint(100, 255) for _ in range(3)]
            img[y:y+bh, x:x+bw] = color
        # Gaussian noise
        img = np.clip(
            img.astype(np.float32) + np.random.randn(h, w, 3) * 10,
            0, 255
        ).astype(np.uint8)
        return img

    def _make_lidar(self):
        """Random LiDAR point cloud (N x 4): x,y,z,intensity."""
        n = random.randint(5000, self.cfg.max_lidar_pts)
        pts = np.random.randn(n, 4).astype(np.float32)
        pts[:, 0] *= 25   # x: ±25 m
        pts[:, 1] *= 25   # y: ±25 m
        pts[:, 2] = np.abs(pts[:, 2]) * 1.5 - 1.5  # z: ground-ish
        pts[:, 3] = np.clip(np.abs(pts[:, 3]), 0, 1)  # intensity
        return pts

    def _make_boxes(self, max_boxes: int = 15):
        """Random 2D bounding boxes (pascal_voc) and class labels."""
        n = random.randint(1, max_boxes)
        h, w = self.cfg.img_h, self.cfg.img_w
        boxes, labels = [], []
        for _ in range(n):
            x1 = random.randint(0, w - 50)
            y1 = random.randint(h // 3, h - 40)
            x2 = min(x1 + random.randint(30, 120), w - 1)
            y2 = min(y1 + random.randint(20, 80),  h - 1)
            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2])
                labels.append(random.randint(0, self.cfg.num_classes - 1))
        if not boxes:
            boxes = [[10, h // 2, 60, h // 2 + 40]]
            labels = [0]
        return boxes, labels

    def _make_seg_mask(self):
        """Coarse semantic segmentation mask."""
        h, w = self.cfg.img_h, self.cfg.img_w
        mask = np.zeros((h, w), dtype=np.int64)
        # Sky
        mask[:h // 3, :] = 5
        # Road
        mask[2 * h // 3:, :] = 0
        # Lane markers
        for x in [w // 4, w // 2, 3 * w // 4]:
            mask[2 * h // 3:, max(0, x-3):min(w, x+3)] = 1
        # Buildings
        mask[h // 3: 2 * h // 3, :w // 4] = 3
        mask[h // 3: 2 * h // 3, 3 * w // 4:] = 3
        return mask

    def __getitem__(self, idx):
        cfg = self.cfg

        # 1. Raw image & weather augmentation
        img = self._make_image()
        if self.train:
            img, weather_cond = WEATHER_AUG(img, cfg)
        else:
            weather_cond = 'clear'

        # 2. Ground truth boxes & labels
        boxes, class_labels = self._make_boxes()

        # 3. Albumentations
        result = self.transforms(
            image=img, bboxes=boxes, class_labels=class_labels
        )
        img_t = result['image']          # (3, H, W) float32
        bboxes = result['bboxes']
        labels = result['class_labels']

        # Normalise boxes to [0,1]
        h, w = cfg.img_h, cfg.img_w
        if bboxes:
            boxes_norm = torch.tensor(
                [[x1/w, y1/h, x2/w, y2/h] for x1,y1,x2,y2 in bboxes],
                dtype=torch.float32
            )
            labels_t = torch.tensor(labels, dtype=torch.long)
        else:
            boxes_norm = torch.zeros((1, 4), dtype=torch.float32)
            labels_t   = torch.zeros(1, dtype=torch.long)

        # 4. LiDAR
        pts = self._make_lidar()
        if self.train:
            pts = augment_lidar_weather(pts, weather_cond)

        # Fixed-size LiDAR tensor via random subsample / pad
        N = cfg.max_lidar_pts
        if len(pts) >= N:
            idx_pts = np.random.choice(len(pts), N, replace=False)
            pts = pts[idx_pts]
        else:
            pad = np.zeros((N - len(pts), 4), dtype=np.float32)
            pts = np.vstack([pts, pad])
        lidar_t = torch.from_numpy(pts.astype(np.float32))

        # 5. Segmentation mask
        seg_mask = torch.tensor(self._make_seg_mask(), dtype=torch.long)

        # 6. Weather label (integer)
        cond_map = {'clear':0,'fog':1,'rain':2,'snow':3,'night':4,'dust':5}
        weather_t = torch.tensor(cond_map.get(weather_cond, 0), dtype=torch.long)

        # 7. Synthetic future trajectory (5 waypoints)
        trajectory = torch.randn(5, 2) * 2.0  # Δx, Δy in metres

        return {
            'image'     : img_t,
            'lidar'     : lidar_t,
            'boxes'     : boxes_norm,
            'labels'    : labels_t,
            'seg_mask'  : seg_mask,
            'weather'   : weather_t,
            'trajectory': trajectory,
            'condition' : weather_cond,
        }


def collate_fn(batch):
    """Custom collate to handle variable-length box tensors."""
    images = torch.stack([b['image'] for b in batch])
    lidars = torch.stack([b['lidar'] for b in batch])
    seg_masks = torch.stack([b['seg_mask'] for b in batch])
    weathers = torch.stack([b['weather'] for b in batch])
    trajectories = torch.stack([b['trajectory'] for b in batch])
    # Boxes are variable → keep as list
    boxes  = [b['boxes']  for b in batch]
    labels = [b['labels'] for b in batch]
    conditions = [b['condition'] for b in batch]
    return {
        'image'     : images,
        'lidar'     : lidars,
        'boxes'     : boxes,
        'labels'    : labels,
        'seg_mask'  : seg_masks,
        'weather'   : weathers,
        'trajectory': trajectories,
        'condition' : conditions,
    }


# ── Instantiate loaders ───────────────────────────────────────
TRAIN_DS = WaymoSyntheticDataset(CFG, 'train', size=400)
VAL_DS   = WaymoSyntheticDataset(CFG, 'val',   size=100)

TRAIN_LOADER = DataLoader(
    TRAIN_DS, batch_size=CFG.batch_size,
    shuffle=True,  collate_fn=collate_fn,
    num_workers=CFG.num_workers, pin_memory=True, drop_last=True
)
VAL_LOADER = DataLoader(
    VAL_DS,   batch_size=CFG.batch_size,
    shuffle=False, collate_fn=collate_fn,
    num_workers=CFG.num_workers, pin_memory=True
)

print(f'Train samples: {len(TRAIN_DS)} | {len(TRAIN_LOADER)} batches')
print(f'Val   samples: {len(VAL_DS)}   | {len(VAL_LOADER)} batches')

# Quick sanity check
sample = next(iter(TRAIN_LOADER))
print(f'Image batch   : {sample["image"].shape}')
print(f'LiDAR batch   : {sample["lidar"].shape}')
print(f'Seg mask batch: {sample["seg_mask"].shape}')
print(f'Weather labels: {sample["weather"]}')

Train samples: 400 | 100 batches
Val   samples: 100   | 25 batches
Image batch   : torch.Size([4, 3, 384, 640])
LiDAR batch   : torch.Size([4, 20000, 4])
Seg mask batch: torch.Size([4, 384, 640])
Weather labels: tensor([4, 1, 1, 3])


## Cell 6 — PointPillars LiDAR Encoder

In [7]:

# CELL 6: PointPillars LiDAR Encoder
#
# Reference: Lang et al. (2019) "PointPillars: Fast Encoders
#            for Object Detection from Point Clouds"
#
# Steps:
#   1. Voxelise point cloud into vertical pillars
#   2. Per-pillar PointNet feature extraction
#   3. Scatter features to BEV pseudo-image

class PillarFeatureNet(nn.Module):
    """
    Per-pillar PointNet: maps (N_pts x in_dim) → (out_dim,)
    """
    def __init__(self, in_dim: int = 9, out_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU(),
        )
        self.out_dim = out_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B * N_pillars * N_pts, in_dim)
        return self.net(x)


class PointPillarsEncoder(nn.Module):
    """
    Full PointPillars encoder.
    Input : (B, N_pts, 4) raw LiDAR  [x, y, z, intensity]
    Output: (B, out_channels, BEV_H, BEV_W)
    """

    def __init__(self, cfg: Config, out_channels: int = 256):
        super().__init__()
        self.cfg = cfg
        self.bev_h = cfg.bev_h
        self.bev_w = cfg.bev_w

        pcr = cfg.point_cloud_range
        vs  = cfg.voxel_size
        self.x_min, self.y_min = pcr[0], pcr[1]
        self.x_max, self.y_max = pcr[3], pcr[4]
        self.vx = vs[0]
        self.vy = vs[1]

        self.pillar_feat = PillarFeatureNet(in_dim=7, out_dim=64)

        # BEV backbone (simple CNN)
        self.bev_backbone = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, out_channels, 3, padding=1), nn.BatchNorm2d(out_channels), nn.ReLU(),
        )
        self.out_channels = out_channels

    def _voxelise(self, pts: torch.Tensor):
        """
        Quick differentiable voxelisation via grid_sample.
        For speed in this demo, we project points into a BEV
        density image and use max-pooled intensity.
        pts: (B, N, 4)
        returns: (B, 1, BEV_H, BEV_W) — density image
        """
        B, N, _ = pts.shape
        H, W = self.bev_h, self.bev_w

        # Normalise x,y to [−1,+1] for grid_sample
        x  = pts[..., 0]  # (B, N)
        y  = pts[..., 1]
        ix = ((x - self.x_min) / (self.x_max - self.x_min) * (W - 1))
        iy = ((y - self.y_min) / (self.y_max - self.y_min) * (H - 1))

        # Clamp to grid
        ix = ix.long().clamp(0, W - 1)
        iy = iy.long().clamp(0, H - 1)

        # Scatter: count points per cell (density) + mean z
        # Using two feature channels: density, mean_z
        device = pts.device
        bev = torch.zeros(B, 2, H, W, device=device)
        cnt = torch.zeros(B, 1, H, W, device=device)

        for b in range(B):
            flat = iy[b] * W + ix[b]  # (N,)
            # density
            ones = torch.ones(N, device=device)
            bev_flat = torch.zeros(H * W, device=device)
            bev_flat.scatter_add_(0, flat, ones)
            bev[b, 0] = bev_flat.view(H, W)
            # mean z
            z_flat = torch.zeros(H * W, device=device)
            z_flat.scatter_add_(0, flat, pts[b, :, 2])
            count_flat = bev_flat.clamp(min=1)
            bev[b, 1] = (z_flat / count_flat).view(H, W)

        # Normalise density log1p
        bev[:, 0] = torch.log1p(bev[:, 0])
        return bev  # (B, 2, H, W)

    def forward(self, pts: torch.Tensor) -> torch.Tensor:
        # pts: (B, N, 4)
        bev2 = self._voxelise(pts)           # (B, 2, H, W)

        # Expand to 64 channels via learned 1×1 convolutions
        # (using the BEV backbone; expand input first)
        B = pts.shape[0]
        # Repeat channels to meet PillarFeatureNet expectation
        bev64 = bev2.repeat(1, 32, 1, 1)    # (B, 64, H, W)

        out = self.bev_backbone(bev64)        # (B, out_channels, H, W)
        return out


# Smoke-test
_pts  = torch.randn(2, CFG.max_lidar_pts, 4).to(DEVICE)
_enc  = PointPillarsEncoder(CFG).to(DEVICE)
_bev  = _enc(_pts)
print(f'PointPillars output: {_bev.shape}')  # (2, 256, 100, 100)
del _pts, _enc, _bev

PointPillars output: torch.Size([2, 256, 100, 100])


## Cell 7 — Image Encoder (Swin Transformer)

In [9]:
import timm

class SwinImageEncoder(nn.Module):
    """
    Swin-S backbone with ImageNet pre-trained weights.
    Outputs a spatial feature map projected to bev_feat_dim.
    Input : (B, 3, H, W)
    Output: (B, bev_feat_dim, H', W')
    """

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg

        # Load Swin-Small with feature-extraction mode
        self.backbone = timm.create_model(
            cfg.backbone,
            pretrained=True,
            features_only=True,
            out_indices=(3,),
            img_size=(cfg.img_h, cfg.img_w)   # Specify input image size
        )
        # Determine backbone output channels by performing a dummy forward pass
        with torch.no_grad():
            dummy = torch.zeros(1, 3, cfg.img_h, cfg.img_w)
            feats = self.backbone(dummy)
            # Timm often returns features in (B, H, W, C) for ViT/Swin features_only=True
            # Permute to (B, C, H, W) to match Conv2d expectation
            dummy_feature_map = feats[0].permute(0, 3, 1, 2)
            in_ch = dummy_feature_map.shape[1]

        # Project to bev_feat_dim
        self.proj = nn.Sequential(
            nn.Conv2d(in_ch, cfg.bev_feat_dim, 1),
            nn.BatchNorm2d(cfg.bev_feat_dim),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(x)   # list of tensors
        f = feats[0]               # (B, H', W', C_backbone) from timm for Swin
        f = f.permute(0, 3, 1, 2)  # Permute to (B, C_backbone, H', W')
        return self.proj(f)        # (B, bev_feat_dim, H', W')


# Smoke-test
_img = torch.randn(2, 3, CFG.img_h, CFG.img_w).to(DEVICE)
_enc = SwinImageEncoder(CFG).to(DEVICE)
_out = _enc(_img)
print(f'Swin output: {_out.shape}')   # (2, 256, ~12, ~20)
del _img, _enc, _out

Swin output: torch.Size([2, 256, 20, 768])


## Cell 8 — BEVFusion Fusion Module

In [10]:

# CELL 8: BEVFusion Multi-Modal Fusion Module
#
# Reference: Liu et al. (2022) "BEVFusion: Multi-Task
#            Multi-Sensor Fusion with Unified Bird's-Eye View"
#
# Aligns camera BEV features with LiDAR BEV features and
# fuses them via channel-attention + concatenation.

class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, mid), nn.ReLU(),
            nn.Linear(mid, channels), nn.Sigmoid(),
        )

    def forward(self, x):
        avg = self.fc(self.avg_pool(x))
        mx  = self.fc(self.max_pool(x))
        att = (avg + mx).view(*x.shape[:2], 1, 1)
        return x * att


class BEVFusionModule(nn.Module):
    """
    Fuses camera and LiDAR BEV feature maps.

    Both modalities are first resized to (bev_h, bev_w),
    then channel-attended and concatenated, then projected
    back to bev_feat_dim.
    """

    def __init__(self, cfg: Config):
        super().__init__()
        self.bev_h = cfg.bev_h
        self.bev_w = cfg.bev_w
        fd = cfg.bev_feat_dim

        self.cam_att   = ChannelAttention(fd)
        self.lidar_att = ChannelAttention(fd)

        # Fused: 2*fd → fd
        self.fuse = nn.Sequential(
            nn.Conv2d(fd * 2, fd, 3, padding=1),
            nn.BatchNorm2d(fd), nn.ReLU(),
            nn.Conv2d(fd, fd, 3, padding=1),
            nn.BatchNorm2d(fd), nn.ReLU(),
        )

        # Weather-aware gating: scalar gate per channel
        # (learned trust weighting for each modality per condition)
        self.weather_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(fd * 2, fd * 2),
            nn.Sigmoid()
        )

    def forward(self,
                cam_feat: torch.Tensor,
                lidar_feat: torch.Tensor) -> torch.Tensor:
        """
        cam_feat  : (B, fd, H_c, W_c)  — any spatial size
        lidar_feat: (B, fd, H_l, W_l)  — any spatial size
        returns   : (B, fd, bev_h, bev_w)
        """
        # Align both to (bev_h, bev_w)
        cam_bev   = F.interpolate(cam_feat,   (self.bev_h, self.bev_w),
                                  mode='bilinear', align_corners=False)
        lidar_bev = F.interpolate(lidar_feat, (self.bev_h, self.bev_w),
                                  mode='bilinear', align_corners=False)

        # Channel attention per modality
        cam_bev   = self.cam_att(cam_bev)
        lidar_bev = self.lidar_att(lidar_bev)

        # Concatenate
        cat = torch.cat([cam_bev, lidar_bev], dim=1)  # (B, 2*fd, H, W)

        # Weather-aware gating
        gate = self.weather_gate(cat).view(*cat.shape[:2], 1, 1)
        cat  = cat * gate

        # Fuse
        return self.fuse(cat)   # (B, fd, bev_h, bev_w)


# Smoke-test
_cam_f   = torch.randn(2, CFG.bev_feat_dim, 12, 20).to(DEVICE)
_lidar_f = torch.randn(2, CFG.bev_feat_dim, CFG.bev_h, CFG.bev_w).to(DEVICE)
_fuse    = BEVFusionModule(CFG).to(DEVICE)
_fused   = _fuse(_cam_f, _lidar_f)
print(f'BEVFusion output: {_fused.shape}')  # (2, 256, 100, 100)
del _cam_f, _lidar_f, _fuse, _fused

BEVFusion output: torch.Size([2, 256, 100, 100])


## Cell 9 — DETR3D-Style Transformer Detector

In [11]:

# CELL 9: DETR3D-Style Transformer Object Detector
#
# Reference: Wang et al. (2021) "DETR3D: 3D Object Detection
#            from Multi-view Images via 3D-to-2D Queries"
#
# Uses learned object queries to decode bounding boxes and
# class logits from the fused BEV feature map.


class PositionEmbeddingSine2D(nn.Module):
    """2D sinusoidal positional encoding for BEV grids."""

    def __init__(self, feat_dim: int, temperature: int = 10000):
        super().__init__()
        assert feat_dim % 2 == 0, 'feat_dim must be even'
        self.feat_dim = feat_dim
        self.T = temperature

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        device = x.device
        dim = C // 2
        div = torch.arange(0, dim, 2, dtype=torch.float32, device=device)
        div = self.T ** (div / dim)

        ys = torch.arange(H, dtype=torch.float32, device=device).unsqueeze(1) / div
        xs = torch.arange(W, dtype=torch.float32, device=device).unsqueeze(1) / div

        pe_y = torch.cat([ys.sin(), ys.cos()], dim=1)  # (H, dim)
        pe_x = torch.cat([xs.sin(), xs.cos()], dim=1)  # (W, dim)

        # Combine to (C, H, W)
        pe = torch.zeros(C, H, W, device=device)
        pe[:dim] = pe_y.T[:dim, :, None].expand(-1, H, W)
        pe[dim:] = pe_x.T[:C-dim, None, :].expand(-1, H, W)
        return x + pe.unsqueeze(0)


class DETR3DTransformerDecoder(nn.Module):
    """
    Multi-layer cross-attention transformer decoder.
    Learned queries attend to the BEV feature map.
    """

    def __init__(self, cfg: Config):
        super().__init__()
        d = cfg.bev_feat_dim
        self.pos_enc = PositionEmbeddingSine2D(d)

        # Learnable object queries
        self.query_embed = nn.Embedding(cfg.num_queries, d)

        # Standard PyTorch Transformer decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d,
            nhead=cfg.num_heads,
            dim_feedforward=cfg.ffn_dim,
            dropout=0.1,
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(
            decoder_layer, num_layers=cfg.num_transformer_layers
        )

        # Output heads
        self.cls_head = nn.Linear(d, cfg.num_classes + 1)  # +1 for 'no-object'
        self.box_head = nn.Sequential(
            nn.Linear(d, d // 2), nn.ReLU(),
            nn.Linear(d // 2, 4), nn.Sigmoid()  # normalised cx,cy,w,h
        )

    def forward(self, bev: torch.Tensor):
        """
        bev: (B, C, H, W)
        returns:
          pred_logits : (B, num_queries, num_classes+1)
          pred_boxes  : (B, num_queries, 4)  — cx,cy,w,h in [0,1]
          query_feats : (B, num_queries, C)  — for temporal GRU
        """
        B, C, H, W = bev.shape

        # Positional encoding + flatten to sequence
        mem = self.pos_enc(bev)                       # (B, C, H, W)
        mem = rearrange(mem, 'b c h w -> b (h w) c')  # (B, H*W, C)

        # Repeat queries across batch
        queries = self.query_embed.weight.unsqueeze(0).expand(B, -1, -1)  # (B, Q, C)

        # Transformer decoding
        out = self.decoder(queries, mem)   # (B, Q, C)

        pred_logits = self.cls_head(out)   # (B, Q, K+1)
        pred_boxes  = self.box_head(out)   # (B, Q, 4)
        return pred_logits, pred_boxes, out


# Smoke-test
_bev_f = torch.randn(2, CFG.bev_feat_dim, CFG.bev_h, CFG.bev_w).to(DEVICE)
_det   = DETR3DTransformerDecoder(CFG).to(DEVICE)
_logits, _boxes, _qf = _det(_bev_f)
print(f'Logits: {_logits.shape}  Boxes: {_boxes.shape}')  # (2, 300, 11) (2, 300, 4)
del _bev_f, _det, _logits, _boxes, _qf

Logits: torch.Size([2, 300, 11])  Boxes: torch.Size([2, 300, 4])


## Cell 10 — Semantic Segmentation & GRU Temporal Head

In [12]:
# ============================================================
# CELL 10: Semantic Segmentation Head + GRU Temporal Decoder
# ============================================================

class SegmentationHead(nn.Module):
    """
    Lightweight decoder for semantic segmentation.
    Upsamples BEV features to (img_h, img_w) and predicts
    per-pixel class logits.
    """

    def __init__(self, cfg: Config):
        super().__init__()
        fd = cfg.bev_feat_dim

        self.decoder = nn.Sequential(
            # Progressive upsampling
            nn.Conv2d(fd, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(128, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(64, 32, 3, padding=1),   nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(32, cfg.num_seg_classes, 1),  # final pixel-wise prediction
        )
        self.target_size = (cfg.img_h, cfg.img_w)

    def forward(self, bev: torch.Tensor) -> torch.Tensor:
        out = self.decoder(bev)   # (B, num_seg, H', W')
        return F.interpolate(out, self.target_size,
                             mode='bilinear', align_corners=False)


class TemporalGRUDecoder(nn.Module):
    """
    Predicts future trajectory waypoints using a GRU
    conditioned on transformer query features.

    Input : query_feats (B, Q, C)  — pooled to (B, C)
    Output: trajectory  (B, T, 2)  — T future Δx, Δy waypoints
    """

    def __init__(self, cfg: Config, num_waypoints: int = 5):
        super().__init__()
        self.T = num_waypoints
        d = cfg.bev_feat_dim

        self.pool = nn.AdaptiveAvgPool1d(1)     # (B, C, Q) → (B, C)
        self.gru  = nn.GRU(
            input_size=d, hidden_size=cfg.gru_hidden,
            num_layers=2, batch_first=True, dropout=0.1
        )
        self.out  = nn.Linear(cfg.gru_hidden, 2)  # Δx, Δy

    def forward(self, query_feats: torch.Tensor) -> torch.Tensor:
        # query_feats: (B, Q, C)
        B, Q, C = query_feats.shape

        # Pool over queries to get single context vector
        ctx = self.pool(query_feats.permute(0, 2, 1)).squeeze(-1)  # (B, C)

        # Repeat context as GRU input for T steps
        gru_in = ctx.unsqueeze(1).expand(-1, self.T, -1)   # (B, T, C)
        gru_out, _ = self.gru(gru_in)                       # (B, T, hidden)
        return self.out(gru_out)                             # (B, T, 2)


class WeatherClassifier(nn.Module):
    """Small head to predict current weather condition (auxiliary task)."""

    def __init__(self, cfg: Config, num_conditions: int = 6):
        super().__init__()
        fd = cfg.bev_feat_dim
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(fd, 64), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_conditions),
        )

    def forward(self, bev: torch.Tensor) -> torch.Tensor:
        return self.head(bev)   # (B, num_conditions)


print('Segmentation head, GRU temporal decoder, and weather classifier defined.')

Segmentation head, GRU temporal decoder, and weather classifier defined.


## Cell 11 — Full Model (WeatherRobustDetector)

In [13]:
# ============================================================
# CELL 11: Full WeatherRobustDetector Model
#
# Integrates all modules into a single nn.Module:
#   Swin → PointPillars → BEVFusion → DETR3D
#       → SegHead / GRU / WeatherHead
# ============================================================

class WeatherRobustDetector(nn.Module):
    """
    Multi-modal, weather-robust obstacle detection model.

    Forward inputs:
      image  : (B, 3, H, W)
      lidar  : (B, N, 4)

    Forward outputs (dict):
      pred_logits   : (B, Q, K+1)   — class logits per query
      pred_boxes    : (B, Q, 4)     — normalised cx,cy,w,h
      pred_seg      : (B, S, H, W)  — segmentation logits
      pred_traj     : (B, T, 2)     — trajectory waypoints
      pred_weather  : (B, W_c)      — weather class logits
      bev_feat      : (B, fd, bH, bW) — BEV feature (for viz)
    """

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg

        # Encoders
        self.image_encoder  = SwinImageEncoder(cfg)
        self.lidar_encoder  = PointPillarsEncoder(cfg, cfg.bev_feat_dim)

        # Fusion
        self.fusion         = BEVFusionModule(cfg)

        # Detection transformer
        self.detector       = DETR3DTransformerDecoder(cfg)

        # Auxiliary heads
        self.seg_head       = SegmentationHead(cfg)
        self.traj_decoder   = TemporalGRUDecoder(cfg)
        self.weather_head   = WeatherClassifier(cfg)

        # Initialise non-pretrained weights
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self,
                image: torch.Tensor,
                lidar: torch.Tensor) -> Dict[str, torch.Tensor]:

        # 1. Image encoding
        cam_feat  = self.image_encoder(image)    # (B, fd, H', W')

        # 2. LiDAR encoding
        lidar_bev = self.lidar_encoder(lidar)    # (B, fd, bH, bW)

        # 3. BEV fusion
        bev = self.fusion(cam_feat, lidar_bev)   # (B, fd, bH, bW)

        # 4. Detection
        pred_logits, pred_boxes, query_feats = self.detector(bev)

        # 5. Segmentation
        pred_seg = self.seg_head(bev)

        # 6. Trajectory
        pred_traj = self.traj_decoder(query_feats)

        # 7. Weather classification (auxiliary)
        pred_weather = self.weather_head(bev)

        return {
            'pred_logits' : pred_logits,
            'pred_boxes'  : pred_boxes,
            'pred_seg'    : pred_seg,
            'pred_traj'   : pred_traj,
            'pred_weather': pred_weather,
            'bev_feat'    : bev,
        }

    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Instantiate and check
MODEL = WeatherRobustDetector(CFG).to(DEVICE)
print(f'Total trainable parameters: {MODEL.count_params():,}')

# Full forward pass smoke-test
_img   = torch.randn(2, 3, CFG.img_h, CFG.img_w).to(DEVICE)
_lidar = torch.randn(2, CFG.max_lidar_pts, 4).to(DEVICE)
with torch.no_grad():
    _out = MODEL(_img, _lidar)
for k, v in _out.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k:15s}: {tuple(v.shape)}')
del _img, _lidar, _out

Total trainable parameters: 59,041,257
  pred_logits    : (2, 300, 11)
  pred_boxes     : (2, 300, 4)
  pred_seg       : (2, 8, 384, 640)
  pred_traj      : (2, 5, 2)
  pred_weather   : (2, 6)
  bev_feat       : (2, 256, 100, 100)


## Cell 12 — Hungarian Matching + Loss Functions

In [14]:
# ============================================================
# CELL 12: Loss Functions
#
# Detection: Hungarian matching (bipartite) + focal CE + GIoU
# Segmentation: Weighted cross-entropy
# Trajectory: Smooth-L1 regression
# Weather: Standard cross-entropy (auxiliary)
# ============================================================

def box_cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    """Convert (cx,cy,w,h) → (x1,y1,x2,y2)."""
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)


def generalised_iou(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """
    Compute GIoU for paired boxes in (x1,y1,x2,y2) format.
    Both tensors: (N, 4)
    """
    # Intersection
    ix1 = torch.max(pred[:, 0], target[:, 0])
    iy1 = torch.max(pred[:, 1], target[:, 1])
    ix2 = torch.min(pred[:, 2], target[:, 2])
    iy2 = torch.min(pred[:, 3], target[:, 3])
    inter = (ix2 - ix1).clamp(0) * (iy2 - iy1).clamp(0)

    # Union
    area_p = (pred[:, 2]   - pred[:, 0])   * (pred[:, 3]   - pred[:, 1])
    area_t = (target[:, 2] - target[:, 0]) * (target[:, 3] - target[:, 1])
    union  = area_p + area_t - inter

    iou = inter / (union + 1e-6)

    # Enclosing box
    ex1 = torch.min(pred[:, 0], target[:, 0])
    ey1 = torch.min(pred[:, 1], target[:, 1])
    ex2 = torch.max(pred[:, 2], target[:, 2])
    ey2 = torch.max(pred[:, 3], target[:, 3])
    enc = (ex2 - ex1).clamp(0) * (ey2 - ey1).clamp(0)

    giou = iou - (enc - union) / (enc + 1e-6)
    return giou


class HungarianMatcher(nn.Module):
    """Optimal bipartite matching between predicted queries and GT boxes."""

    def __init__(self, cost_cls: float = 1.0,
                 cost_box: float = 5.0,
                 cost_giou: float = 2.0):
        super().__init__()
        self.cost_cls  = cost_cls
        self.cost_box  = cost_box
        self.cost_giou = cost_giou

    @torch.no_grad()
    def forward(self, pred_logits, pred_boxes, gt_boxes_list, gt_labels_list):
        """
        Returns list of (pred_idx, gt_idx) tuples per batch element.
        """
        B, Q, _ = pred_logits.shape
        indices = []

        for b in range(B):
            gt_b = gt_boxes_list[b].to(pred_logits.device)  # (M, 4)
            lb_b = gt_labels_list[b].to(pred_logits.device) # (M,)
            M = gt_b.shape[0]

            if M == 0:
                indices.append(([], []))
                continue

            # Classification cost (negative probability of GT class)
            p = pred_logits[b].softmax(-1)  # (Q, K+1)
            cost_cls = -p[:, lb_b]          # (Q, M)

            # L1 box cost
            cost_box = torch.cdist(
                pred_boxes[b], gt_b, p=1
            )  # (Q, M)

            # GIoU cost (convert cx,cy,w,h → xyxy)
            pb_xyxy = box_cxcywh_to_xyxy(pred_boxes[b])  # (Q, 4)
            gb_xyxy = box_cxcywh_to_xyxy(gt_b)           # (M, 4)

            giou_mat = torch.zeros(Q, M, device=pred_logits.device)
            for m in range(M):
                giou_mat[:, m] = generalised_iou(
                    pb_xyxy, gb_xyxy[m].unsqueeze(0).expand(Q, -1)
                )
            cost_giou = -giou_mat

            C = (self.cost_cls * cost_cls +
                 self.cost_box * cost_box +
                 self.cost_giou * cost_giou).cpu().numpy()

            row_idx, col_idx = linear_sum_assignment(C)
            indices.append((row_idx.tolist(), col_idx.tolist()))

        return indices


class DetectionLoss(nn.Module):
    """DETR-style set-prediction loss."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg     = cfg
        self.matcher = HungarianMatcher(
            cfg.loss_cls_w, cfg.loss_box_w, cfg.loss_giou_w
        )
        self.num_classes = cfg.num_classes
        # Class frequency weighting (no-object class gets lower weight)
        empty_w = torch.ones(cfg.num_classes + 1)
        empty_w[-1] = 0.1
        self.register_buffer('empty_w', empty_w)

    def forward(self, pred_logits, pred_boxes,
                gt_boxes_list, gt_labels_list):
        B = pred_logits.shape[0]
        indices = self.matcher(pred_logits, pred_boxes,
                               gt_boxes_list, gt_labels_list)

        # Build target tensors
        target_cls  = torch.full(
            (B, pred_logits.shape[1]), self.num_classes,
            dtype=torch.long, device=pred_logits.device
        )  # default = no-object

        loss_box  = torch.tensor(0.0, device=pred_logits.device)
        loss_giou = torch.tensor(0.0, device=pred_logits.device)
        n_matched = 0

        for b, (pred_idx, gt_idx) in enumerate(indices):
            if len(pred_idx) == 0:
                continue
            gt_b = gt_boxes_list[b].to(pred_logits.device)
            lb_b = gt_labels_list[b].to(pred_logits.device)

            target_cls[b, pred_idx] = lb_b[gt_idx]

            # Box regression (L1)
            pb = pred_boxes[b][pred_idx]   # (K, 4)
            gb = gt_b[gt_idx]              # (K, 4)
            loss_box += F.l1_loss(pb, gb, reduction='sum')

            # GIoU
            pb_xy = box_cxcywh_to_xyxy(pb)
            gb_xy = box_cxcywh_to_xyxy(gb)
            loss_giou += (1 - generalised_iou(pb_xy, gb_xy)).sum()

            n_matched += len(pred_idx)

        n = max(n_matched, 1)
        loss_cls  = F.cross_entropy(pred_logits.flatten(0, 1),
                                    target_cls.flatten(),
                                    weight=self.empty_w)
        return {
            'loss_cls' : loss_cls,
            'loss_box' : loss_box  / n,
            'loss_giou': loss_giou / n,
        }


class TotalLoss(nn.Module):
    """Weighted sum of all task losses."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.det_loss  = DetectionLoss(cfg)
        self.cfg = cfg

    def forward(self, preds: Dict, targets: Dict):
        c = self.cfg

        # Detection
        det = self.det_loss(
            preds['pred_logits'], preds['pred_boxes'],
            targets['boxes'],    targets['labels']
        )
        L_det = (c.loss_cls_w  * det['loss_cls'] +
                 c.loss_box_w  * det['loss_box'] +
                 c.loss_giou_w * det['loss_giou'])

        # Segmentation
        seg_pred = F.interpolate(
            preds['pred_seg'],
            targets['seg_mask'].shape[-2:],
            mode='bilinear', align_corners=False
        )
        L_seg = F.cross_entropy(seg_pred, targets['seg_mask'])

        # Trajectory
        L_traj = F.smooth_l1_loss(preds['pred_traj'],
                                   targets['trajectory'])

        # Weather (auxiliary)
        L_weather = F.cross_entropy(preds['pred_weather'],
                                    targets['weather'])

        total = (L_det +
                 c.loss_seg_w  * L_seg +
                 c.loss_traj_w * L_traj +
                 0.2            * L_weather)

        return total, {
            'det'    : L_det.item(),
            'seg'    : L_seg.item(),
            'traj'   : L_traj.item(),
            'weather': L_weather.item(),
            'total'  : total.item(),
        }


CRITERION = TotalLoss(CFG).to(DEVICE)
print('Loss module ready.')

Loss module ready.


## Cell 13 — Optimizer, Scheduler & Early Stopping

In [15]:
# ============================================================
# CELL 13: Optimizer, LR Scheduler & Early Stopping
# ============================================================

# ── Optimizer: AdamW with weight-decay on non-norm params ────
def build_optimizer(model: nn.Module, cfg: Config):
    # Separate backbone (lower LR) vs new heads (higher LR)
    backbone_params = list(model.image_encoder.backbone.parameters())
    backbone_ids    = set(id(p) for p in backbone_params)
    other_params    = [p for p in model.parameters()
                       if id(p) not in backbone_ids]
    return torch.optim.AdamW(
        [
            {'params': backbone_params, 'lr': cfg.lr * 0.1},
            {'params': other_params,    'lr': cfg.lr},
        ],
        weight_decay=cfg.weight_decay,
    )


OPTIMIZER = build_optimizer(MODEL, CFG)


# ── Cosine LR with warmup ─────────────────────────────────────
def warmup_cosine_schedule(step: int, warmup: int, total: int) -> float:
    if step < warmup:
        return step / max(1, warmup)
    progress = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

total_steps  = CFG.num_epochs * len(TRAIN_LOADER)
warmup_steps = CFG.lr_warmup_epochs * len(TRAIN_LOADER)

SCHEDULER = torch.optim.lr_scheduler.LambdaLR(
    OPTIMIZER,
    lr_lambda=lambda s: warmup_cosine_schedule(s, warmup_steps, total_steps)
)


# ── Mixed precision scaler ────────────────────────────────────
SCALER = GradScaler(enabled=CFG.mixed_precision)


# ── Early stopping ────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience: int = 10, min_delta: float = 1e-4,
                 mode: str = 'min'):
        self.patience   = patience
        self.min_delta  = min_delta
        self.mode       = mode
        self.best       = float('inf') if mode == 'min' else float('-inf')
        self.counter    = 0
        self.best_epoch = 0

    def __call__(self, metric: float, epoch: int) -> bool:
        improved = (
            metric < self.best - self.min_delta if self.mode == 'min'
            else metric > self.best + self.min_delta
        )
        if improved:
            self.best       = metric
            self.counter    = 0
            self.best_epoch = epoch
        else:
            self.counter += 1
        return self.counter >= self.patience


EARLY_STOP = EarlyStopping(CFG.early_stop_patience, mode='min')
print(f'Optimizer, scheduler, and early stopping ready.')
print(f'Total steps: {total_steps}  |  Warmup: {warmup_steps}')

Optimizer, scheduler, and early stopping ready.
Total steps: 5000  |  Warmup: 500


## Cell 14 — Training Loop

In [ ]:
# ============================================================
# CELL 14: Training Loop
#
# Features:
#   • Mixed-precision (AMP)
#   • Gradient accumulation (CFG.grad_accum_steps)
#   • Per-batch LR warmup → cosine decay
#   • TensorBoard + JSON log
#   • Best-checkpoint saving
#   • Early stopping
# ============================================================

from torch.utils.tensorboard import SummaryWriter

LOG_DIR = ROOT / 'logs'
CKPT_DIR = ROOT / 'checkpoints'
WRITER = SummaryWriter(log_dir=str(LOG_DIR))


def run_epoch(model, loader, criterion, optimizer, scaler,
              scheduler, cfg, train: bool):
    model.train() if train else model.eval()
    total_loss = defaultdict(float)
    n_batches  = len(loader)
    prefix     = 'Train' if train else 'Val'

    with tqdm(loader, desc=prefix, leave=False) as pbar:
        for step, batch in enumerate(pbar):
            imgs   = batch['image'].to(DEVICE, non_blocking=True)
            lidars = batch['lidar'].to(DEVICE, non_blocking=True)
            seg_gt = batch['seg_mask'].to(DEVICE, non_blocking=True)
            traj_gt= batch['trajectory'].to(DEVICE, non_blocking=True)
            w_gt   = batch['weather'].to(DEVICE, non_blocking=True)

            targets = {
                'boxes'    : batch['boxes'],
                'labels'   : batch['labels'],
                'seg_mask' : seg_gt,
                'trajectory': traj_gt,
                'weather'  : w_gt,
            }

            ctx = autocast(enabled=cfg.mixed_precision)

            if train:
                with ctx:
                    preds = model(imgs, lidars)
                    loss, loss_dict = criterion(preds, targets)
                    loss = loss / cfg.grad_accum_steps

                scaler.scale(loss).backward()

                if (step + 1) % cfg.grad_accum_steps == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()
            else:
                with torch.no_grad(), ctx:
                    preds = model(imgs, lidars)
                    _, loss_dict = criterion(preds, targets)

            for k, v in loss_dict.items():
                total_loss[k] += v

            pbar.set_postfix(loss=f"{loss_dict['total']:.3f}",
                             lr=f"{scheduler.get_last_lr()[0]:.2e}")

    return {k: v / n_batches for k, v in total_loss.items()}


def save_checkpoint(model, optimizer, epoch, loss, best=False):
    fname = 'best.pth' if best else f'epoch_{epoch:03d}.pth'
    path  = CKPT_DIR / fname
    torch.save({
        'epoch'     : epoch,
        'state_dict': model.state_dict(),
        'optimizer' : optimizer.state_dict(),
        'loss'      : loss,
        'cfg'       : CFG,
    }, path)
    return path


# ── Main training loop ────────────────────────────────────────
TRAIN_HISTORY = []
best_val_loss = float('inf')

print('=' * 60)
print('Starting training...')
print(f'Epochs: {CFG.num_epochs}  |  Batch: {CFG.batch_size}  |  AMP: {CFG.mixed_precision}')
print('=' * 60)

for epoch in range(1, CFG.num_epochs + 1):
    t0 = time.time()

    train_metrics = run_epoch(
        MODEL, TRAIN_LOADER, CRITERION, OPTIMIZER,
        SCALER, SCHEDULER, CFG, train=True
    )
    val_metrics = run_epoch(
        MODEL, VAL_LOADER, CRITERION, OPTIMIZER,
        SCALER, SCHEDULER, CFG, train=False
    )

    elapsed = time.time() - t0
    lr_now  = SCHEDULER.get_last_lr()[0]

    log_entry = {
        'epoch': epoch,
        'train': train_metrics,
        'val'  : val_metrics,
        'lr'   : lr_now,
        'time' : elapsed,
    }
    TRAIN_HISTORY.append(log_entry)

    # TensorBoard
    for k, v in train_metrics.items():
        WRITER.add_scalar(f'train/{k}', v, epoch)
    for k, v in val_metrics.items():
        WRITER.add_scalar(f'val/{k}', v, epoch)
    WRITER.add_scalar('lr', lr_now, epoch)

    # Console log
    print(f'Epoch {epoch:3d}/{CFG.num_epochs} '
          f'| Train {train_metrics["total"]:.3f} '
          f'| Val {val_metrics["total"]:.3f} '
          f'| LR {lr_now:.2e} '
          f'| {elapsed:.1f}s')

    # Checkpoint: best model
    if val_metrics['total'] < best_val_loss:
        best_val_loss = val_metrics['total']
        save_checkpoint(MODEL, OPTIMIZER, epoch, best_val_loss, best=True)
        print(f'  ✓ New best val loss: {best_val_loss:.4f}')

    # Periodic checkpoint every 10 epochs
    if epoch % 10 == 0:
        save_checkpoint(MODEL, OPTIMIZER, epoch, val_metrics['total'])

    # Early stopping
    if EARLY_STOP(val_metrics['total'], epoch):
        print(f'Early stopping at epoch {epoch} '
              f'(best epoch: {EARLY_STOP.best_epoch})')
        break

# Save training history JSON
with open(ROOT / 'logs' / 'train_history.json', 'w') as f:
    json.dump(TRAIN_HISTORY, f, indent=2)

WRITER.close()
print('\nTraining complete.')

Starting training...
Epochs: 50  |  Batch: 4  |  AMP: True


Train:   0%|          | 0/100 [00:00<?, ?it/s]

Val:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch   1/50 | Train 13.055 | Val 8.182 | LR 5.00e-07 | 72.8s
  ✓ New best val loss: 8.1817


Train:   0%|          | 0/100 [00:00<?, ?it/s]

...

Epoch  18/50 | Train 4.615 | Val 4.580 | LR 9.00e-06 | 69.6s
  ✓ New best val loss: 4.5799


Train:   0%|          | 0/100 [00:00<?, ?it/s]

Val:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch  19/50 | Train 4.457 | Val 4.429 | LR 9.50e-06 | 78.5s
  ✓ New best val loss: 4.4290


Train:   0%|          | 0/100 [00:00<?, ?it/s]

## Cell 15 — Evaluation Metrics

In [ ]:

# CELL 15: Comprehensive Evaluation Metrics
#   mAP, IoU, Precision, Recall, F1, Weather Robustness Score

def box_iou(pred_boxes: torch.Tensor, gt_boxes: torch.Tensor) -> torch.Tensor:
    """
    Compute IoU matrix: (N, M) for N pred boxes vs M gt boxes.
    Boxes in (x1, y1, x2, y2) format.
    """
    N, M = len(pred_boxes), len(gt_boxes)
    iou_mat = torch.zeros(N, M)
    for i in range(N):
        for j in range(M):
            ix1 = max(pred_boxes[i,0], gt_boxes[j,0])
            iy1 = max(pred_boxes[i,1], gt_boxes[j,1])
            ix2 = min(pred_boxes[i,2], gt_boxes[j,2])
            iy2 = min(pred_boxes[i,3], gt_boxes[j,3])
            inter = max(0, ix2-ix1) * max(0, iy2-iy1)
            area_p = (pred_boxes[i,2]-pred_boxes[i,0]) * (pred_boxes[i,3]-pred_boxes[i,1])
            area_g = (gt_boxes[j,2]-gt_boxes[j,0]) * (gt_boxes[j,3]-gt_boxes[j,1])
            iou_mat[i,j] = inter / (area_p + area_g - inter + 1e-6)
    return iou_mat


class MetricTracker:
    """
    Accumulates predictions and ground truths across the
    validation set, then computes all metrics in one call.
    """

    def __init__(self, cfg: Config, iou_thresh: float = 0.5):
        self.cfg = cfg
        self.iou_thresh = iou_thresh
        self.reset()

    def reset(self):
        self.tp_per_cls  = defaultdict(int)
        self.fp_per_cls  = defaultdict(int)
        self.fn_per_cls  = defaultdict(int)
        self.all_iou     = []
        self.seg_iou     = defaultdict(list)
        self.weather_correct = defaultdict(list)  # cond → [bool]
        self.n_samples   = 0

    def update(self, preds: Dict, targets: Dict, conditions: List[str]):
        B = preds['pred_logits'].shape[0]

        # ── Detection metrics ─────────────────────────────────
        for b in range(B):
            logits = preds['pred_logits'][b]  # (Q, K+1)
            scores, pred_cls = logits.softmax(-1)[:, :-1].max(-1)
            pred_box = preds['pred_boxes'][b]  # (Q, 4)

            # Filter high-confidence predictions
            mask   = scores > 0.3
            p_cls  = pred_cls[mask]
            p_box  = box_cxcywh_to_xyxy(pred_box[mask])

            gt_box = box_cxcywh_to_xyxy(targets['boxes'][b])
            gt_cls = targets['labels'][b]

            gt_matched = torch.zeros(len(gt_box), dtype=torch.bool)

            for pi in range(len(p_cls)):
                c = p_cls[pi].item()
                if len(gt_box) == 0:
                    self.fp_per_cls[c] += 1
                    continue
                iou_row = box_iou(p_box[pi:pi+1], gt_box)[0]  # (M,)
                best_iou, best_j = iou_row.max(0)
                best_iou = best_iou.item()
                self.all_iou.append(best_iou)
                if (best_iou >= self.iou_thresh and
                        gt_cls[best_j].item() == c and
                        not gt_matched[best_j]):
                    self.tp_per_cls[c] += 1
                    gt_matched[best_j] = True
                else:
                    self.fp_per_cls[c] += 1

            for j in range(len(gt_box)):
                if not gt_matched[j]:
                    c = gt_cls[j].item()
                    self.fn_per_cls[c] += 1

        # ── Segmentation IoU ──────────────────────────────────
        seg_pred = preds['pred_seg'].argmax(1)  # (B, H, W)
        seg_gt   = targets['seg_mask']          # (B, H, W)
        seg_pred_rs = F.interpolate(
            seg_pred.float().unsqueeze(1), seg_gt.shape[-2:],
            mode='nearest'
        ).squeeze(1).long()

        for cls in range(self.cfg.num_seg_classes):
            inter = ((seg_pred_rs == cls) & (seg_gt == cls)).sum().item()
            union = ((seg_pred_rs == cls) | (seg_gt == cls)).sum().item()
            if union > 0:
                self.seg_iou[cls].append(inter / union)

        # ── Weather accuracy ──────────────────────────────────
        w_pred = preds['pred_weather'].argmax(1)  # (B,)
        w_gt   = targets['weather']               # (B,)
        for b, cond in enumerate(conditions):
            correct = (w_pred[b].item() == w_gt[b].item())
            self.weather_correct[cond].append(correct)

        self.n_samples += B

    def compute(self) -> Dict:
        results = {}

        # Per-class precision, recall, F1
        all_p, all_r, all_f1, all_ap = [], [], [], []
        for c in range(self.cfg.num_classes):
            tp = self.tp_per_cls[c]
            fp = self.fp_per_cls[c]
            fn = self.fn_per_cls[c]
            prec = tp / (tp + fp + 1e-6)
            rec  = tp / (tp + fn + 1e-6)
            f1   = 2 * prec * rec / (prec + rec + 1e-6)
            ap   = prec * rec   # single-threshold AP approximation
            all_p.append(prec)
            all_r.append(rec)
            all_f1.append(f1)
            all_ap.append(ap)
            results[f'AP_{self.cfg.class_names[c]}'] = round(ap, 4)

        results['mAP']       = round(float(np.mean(all_ap)), 4)
        results['Precision'] = round(float(np.mean(all_p)), 4)
        results['Recall']    = round(float(np.mean(all_r)), 4)
        results['F1']        = round(float(np.mean(all_f1)), 4)
        results['mean_IoU_det'] = round(
            float(np.mean(self.all_iou)) if self.all_iou else 0.0, 4
        )

        # Segmentation mIoU
        seg_ious = [np.mean(v) for v in self.seg_iou.values() if v]
        results['mIoU_seg'] = round(float(np.mean(seg_ious)) if seg_ious else 0.0, 4)

        # Weather robustness score: mean accuracy across conditions
        w_scores = {}
        for cond, bools in self.weather_correct.items():
            w_scores[cond] = round(float(np.mean(bools)), 4)
        results['weather_robustness'] = w_scores
        results['overall_weather_acc'] = round(
            float(np.mean(list(w_scores.values()))) if w_scores else 0.0, 4
        )

        return results


# ── Run evaluation on val set ─────────────────────────────────
print('Evaluating on validation set...')
TRACKER = MetricTracker(CFG)
MODEL.eval()

with torch.no_grad():
    for batch in tqdm(VAL_LOADER, desc='Eval'):
        imgs   = batch['image'].to(DEVICE)
        lidars = batch['lidar'].to(DEVICE)
        w_gt   = batch['weather'].to(DEVICE)
        seg_gt = batch['seg_mask'].to(DEVICE)

        with autocast(enabled=CFG.mixed_precision):
            preds = MODEL(imgs, lidars)

        targets = {
            'boxes'   : batch['boxes'],
            'labels'  : batch['labels'],
            'seg_mask': seg_gt,
            'weather' : w_gt,
        }
        TRACKER.update(preds, targets, batch['condition'])

METRICS = TRACKER.compute()

print('\n' + '='*55)
print('EVALUATION RESULTS')
print('='*55)
print(f"  mAP              : {METRICS['mAP']}")
print(f"  Precision        : {METRICS['Precision']}")
print(f"  Recall           : {METRICS['Recall']}")
print(f"  F1 Score         : {METRICS['F1']}")
print(f"  Det. IoU         : {METRICS['mean_IoU_det']}")
print(f"  Seg. mIoU        : {METRICS['mIoU_seg']}")
print(f"  Weather Acc.     : {METRICS['overall_weather_acc']}")
print('  Weather breakdown:')
for cond, acc in METRICS['weather_robustness'].items():
    print(f'    {cond:8s}: {acc:.3f}')
print('='*55)

# Save metrics JSON
with open(ROOT / 'logs' / 'eval_metrics.json', 'w') as f:
    json.dump(METRICS, f, indent=2)
print('Metrics saved.')

## Cell 16 — Visualisation

In [ ]:

# CELL 16: Comprehensive Visualisation
#   1. Bounding boxes overlay on RGB
#   2. Segmentation mask
#   3. BEV feature map
#   4. Weather condition detection
#   5. Training loss curves
#   6. Weather augmentation gallery


# ── Colour palettes ───────────────────────────────────────────
CLASS_COLORS = plt.cm.tab10(np.linspace(0, 1, CFG.num_classes))
SEG_COLORS   = [
    '#808080', '#FFFF00', '#C8A0C8', '#464646',
    '#6B8E23', '#87CEEB', '#FF4500', '#000000'
]

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

def denorm(t: torch.Tensor) -> np.ndarray:
    """Denormalise ImageNet-normalised tensor to uint8 HWC."""
    img = t.cpu().permute(1, 2, 0).numpy()
    img = img * IMAGENET_STD + IMAGENET_MEAN
    return (np.clip(img, 0, 1) * 255).astype(np.uint8)


def draw_boxes(ax, img: np.ndarray,
               boxes: np.ndarray, scores: np.ndarray, classes: np.ndarray,
               class_names: List[str]):
    """Draw bounding boxes on ax."""
    ax.imshow(img)
    h, w = img.shape[:2]
    for box, score, cls in zip(boxes, scores, classes):
        if score < 0.3:
            continue
        x1, y1, x2, y2 = box
        x1, x2 = x1 * w, x2 * w
        y1, y2 = y1 * h, y2 * h
        c = CLASS_COLORS[cls % len(CLASS_COLORS)]
        rect = mpatches.FancyBboxPatch(
            (x1, y1), x2-x1, y2-y1,
            boxstyle='round,pad=0', linewidth=2, edgecolor=c, facecolor='none'
        )
        ax.add_patch(rect)
        label = f'{class_names[cls]} {score:.2f}'
        ax.text(x1, y1-3, label, color='white', fontsize=7,
                bbox=dict(facecolor=c, alpha=0.7, pad=1, edgecolor='none'))
    ax.axis('off')


def colorise_seg(mask: np.ndarray) -> np.ndarray:
    """Map class IDs → RGB image."""
    h, w = mask.shape
    rgb  = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_idx, hex_col in enumerate(SEG_COLORS):
        r, g, b = int(hex_col[1:3], 16), int(hex_col[3:5], 16), int(hex_col[5:7], 16)
        rgb[mask == cls_idx] = [r, g, b]
    return rgb


def visualise_sample(model, batch, cfg, out_dir: Path, idx: int = 0):
    """Full 6-panel visualisation for one sample."""
    model.eval()
    imgs   = batch['image'][[idx]].to(DEVICE)
    lidars = batch['lidar'][[idx]].to(DEVICE)

    with torch.no_grad(), autocast(enabled=cfg.mixed_precision):
        preds = model(imgs, lidars)

    img_np   = denorm(batch['image'][idx])
    seg_gt   = batch['seg_mask'][idx].numpy()
    cond     = batch['condition'][idx]

    # Predicted boxes
    logits   = preds['pred_logits'][0]
    probs    = logits.softmax(-1)
    scores, classes = probs[:, :-1].max(-1)
    boxes_np = box_cxcywh_to_xyxy(preds['pred_boxes'][0]).cpu().numpy()
    scores_np = scores.cpu().numpy()
    classes_np = classes.cpu().numpy()

    # Predicted seg mask
    seg_pred = preds['pred_seg'][0].argmax(0).cpu().numpy()
    seg_rs   = cv2.resize(seg_pred.astype(np.uint8), (img_np.shape[1], img_np.shape[0]),
                          interpolation=cv2.INTER_NEAREST)

    # BEV feature map
    bev = preds['bev_feat'][0].mean(0).cpu().numpy()  # (H, W)

    # Predicted weather
    w_pred_idx = preds['pred_weather'][0].argmax().item()
    cond_names = ['Clear', 'Fog', 'Rain', 'Snow', 'Night', 'Dust']
    w_probs    = preds['pred_weather'][0].softmax(0).cpu().numpy()

    # Trajectory
    traj = preds['pred_traj'][0].cpu().numpy()  # (5, 2)

    # ── Plot ──────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.patch.set_facecolor('#0d0d0d')
    fig.suptitle(
        f'WeatherRobustDetector — Ground Condition: {cond.upper()}',
        fontsize=14, color='white', fontweight='bold', y=0.98
    )

    for ax in axes.flat:
        ax.set_facecolor('#1a1a1a')

    # Panel 1: RGB + bounding boxes
    draw_boxes(axes[0,0], img_np, boxes_np, scores_np, classes_np, cfg.class_names)
    axes[0,0].set_title('RGB + Predicted Boxes', color='white', fontsize=11)

    # Panel 2: GT segmentation mask
    axes[0,1].imshow(colorise_seg(seg_gt))
    axes[0,1].set_title('GT Segmentation', color='white', fontsize=11)
    axes[0,1].axis('off')
    legend_patches = [mpatches.Patch(color=SEG_COLORS[i], label=n)
                      for i, n in enumerate(cfg.seg_class_names)]
    axes[0,1].legend(handles=legend_patches, loc='lower right',
                     fontsize=7, facecolor='#2a2a2a', labelcolor='white')

    # Panel 3: Predicted segmentation
    axes[0,2].imshow(colorise_seg(seg_rs))
    axes[0,2].set_title('Predicted Segmentation', color='white', fontsize=11)
    axes[0,2].axis('off')

    # Panel 4: BEV feature map
    im = axes[1,0].imshow(bev, cmap='plasma')
    plt.colorbar(im, ax=axes[1,0], fraction=0.046)
    # Overlay trajectory on BEV
    traj_x = traj[:, 0] / 50 * CFG.bev_w / 2 + CFG.bev_w / 2
    traj_y = traj[:, 1] / 50 * CFG.bev_h / 2 + CFG.bev_h / 2
    axes[1,0].plot(traj_x, traj_y, 'o-', color='cyan', linewidth=2, markersize=6)
    axes[1,0].set_title('BEV Feature Map + Trajectory', color='white', fontsize=11)
    axes[1,0].set_xlabel('X (BEV pixels)', color='grey', fontsize=8)
    axes[1,0].set_ylabel('Y (BEV pixels)', color='grey', fontsize=8)
    axes[1,0].tick_params(colors='grey')

    # Panel 5: Weather probability bar chart
    bars = axes[1,1].bar(cond_names, w_probs,
                         color=['#4caf50' if i == w_pred_idx else '#555' for i in range(6)])
    axes[1,1].set_title('Weather Condition Prediction', color='white', fontsize=11)
    axes[1,1].set_ylim(0, 1)
    axes[1,1].tick_params(colors='grey', rotation=30)
    axes[1,1].set_facecolor('#1a1a1a')
    for spine in axes[1,1].spines.values():
        spine.set_edgecolor('#444')
    axes[1,1].set_ylabel('Probability', color='grey', fontsize=8)
    axes[1,1].text(w_pred_idx, w_probs[w_pred_idx] + 0.02,
                   f'{w_probs[w_pred_idx]:.2f}', ha='center',
                   color='lime', fontsize=9, fontweight='bold')

    # Panel 6: LiDAR BEV density scatter
    pts_np = batch['lidar'][idx].numpy()  # (N, 4)
    valid  = np.abs(pts_np).sum(axis=1) > 0
    pts_v  = pts_np[valid]
    sc = axes[1,2].scatter(
        pts_v[:, 0], pts_v[:, 1],
        c=pts_v[:, 2], cmap='viridis', s=0.5, alpha=0.6
    )
    plt.colorbar(sc, ax=axes[1,2], fraction=0.046, label='Height (m)')
    axes[1,2].set_xlim(-50, 50)
    axes[1,2].set_ylim(-50, 50)
    axes[1,2].set_title('LiDAR Point Cloud (BEV)', color='white', fontsize=11)
    axes[1,2].set_facecolor('#1a1a1a')
    axes[1,2].set_xlabel('X (m)', color='grey', fontsize=8)
    axes[1,2].set_ylabel('Y (m)', color='grey', fontsize=8)
    axes[1,2].tick_params(colors='grey')

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    save_path = out_dir / f'sample_viz_{idx}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print(f'Saved: {save_path}')


def plot_training_curves(history: List[Dict], out_dir: Path):
    """Loss curves for train/val over all epochs."""
    epochs = [h['epoch']          for h in history]
    t_tot  = [h['train']['total'] for h in history]
    v_tot  = [h['val']['total']   for h in history]
    t_seg  = [h['train']['seg']   for h in history]
    v_seg  = [h['val']['seg']     for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0d0d0d')
    for ax in [ax1, ax2]:
        ax.set_facecolor('#1a1a1a')
        for spine in ax.spines.values():
            spine.set_edgecolor('#444')
        ax.tick_params(colors='grey')

    ax1.plot(epochs, t_tot, color='#4fc3f7', linewidth=2, label='Train Total')
    ax1.plot(epochs, v_tot, color='#f48fb1', linewidth=2, label='Val Total',  linestyle='--')
    ax1.set_title('Total Loss', color='white', fontsize=12)
    ax1.set_xlabel('Epoch', color='grey'); ax1.set_ylabel('Loss', color='grey')
    ax1.legend(facecolor='#2a2a2a', labelcolor='white')

    ax2.plot(epochs, t_seg, color='#a5d6a7', linewidth=2, label='Train Seg')
    ax2.plot(epochs, v_seg, color='#ffcc80', linewidth=2, label='Val Seg', linestyle='--')
    ax2.set_title('Segmentation Loss', color='white', fontsize=12)
    ax2.set_xlabel('Epoch', color='grey'); ax2.set_ylabel('Loss', color='grey')
    ax2.legend(facecolor='#2a2a2a', labelcolor='white')

    plt.suptitle('Training Curves', color='white', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_path = out_dir / 'training_curves.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print(f'Saved: {save_path}')


def weather_augmentation_gallery(out_dir: Path):
    """Side-by-side weather augmentation showcase."""
    aug = WeatherAugmentor()
    h, w = 300, 400
    base_img = np.zeros((h, w, 3), dtype=np.uint8)
    for row in range(h):
        t = row / h
        base_img[row] = (
            np.array([100, 150, 200]) * (1 - t) +
            np.array([60,  60,  60])  * t
        ).astype(np.uint8)

    conditions = [
        ('Clear',  base_img.copy()),
        ('Fog',    aug.add_fog(base_img,  0.6)),
        ('Rain',   aug.add_rain(base_img, 250)),
        ('Snow',   aug.add_snow(base_img, 500)),
        ('Night',  aug.add_night(base_img, 2.8)),
        ('Dust',   aug.add_dust(base_img,  0.5)),
    ]

    fig, axes = plt.subplots(1, 6, figsize=(20, 4))
    fig.patch.set_facecolor('#0d0d0d')
    for ax, (name, img) in zip(axes, conditions):
        ax.imshow(img)
        ax.set_title(name, color='white', fontsize=11, fontweight='bold')
        ax.axis('off')
        ax.set_facecolor('#1a1a1a')

    plt.suptitle('Synthetic Weather Augmentation Gallery',
                 color='white', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_path = out_dir / 'weather_gallery.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print(f'Saved: {save_path}')


VIZ_DIR = ROOT / 'outputs' / 'viz'

# Run all visualisations
weather_augmentation_gallery(VIZ_DIR)

val_sample = next(iter(VAL_LOADER))
visualise_sample(MODEL, val_sample, CFG, VIZ_DIR, idx=0)

if TRAIN_HISTORY:
    plot_training_curves(TRAIN_HISTORY, VIZ_DIR)
else:
    print('No training history to plot yet.')

## Cell 17 — Inference Demo

In [ ]:

# CELL 17: Inference Demo
#
# Load best checkpoint and run inference on a single sample.
# Shows latency benchmarking.


def load_checkpoint(model: nn.Module, ckpt_path: str) -> nn.Module:
    """Load saved weights into model."""
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['state_dict'])
    print(f'Loaded checkpoint: {ckpt_path}')
    print(f'  Saved at epoch {ckpt["epoch"]} | Loss {ckpt["loss"]:.4f}')
    return model


best_ckpt = ROOT / 'checkpoints' / 'best.pth'
if best_ckpt.exists():
    MODEL = load_checkpoint(MODEL, str(best_ckpt))
else:
    print('No checkpoint found — using current weights.')

MODEL.eval()


def inference_single(model, img_np: np.ndarray,
                     pts: np.ndarray, cfg: Config) -> Dict:
    """
    Run inference on a single RGB frame + LiDAR point cloud.

    img_np : (H, W, 3) uint8
    pts    : (N, 4)  float32  x,y,z,intensity
    """
    # Pre-process image
    xform = build_transforms(cfg, train=False)
    res   = xform(image=img_np, bboxes=[], class_labels=[])
    img_t = res['image'].unsqueeze(0).to(DEVICE)   # (1, 3, H, W)

    # Pre-process LiDAR
    N = cfg.max_lidar_pts
    if len(pts) >= N:
        idx = np.random.choice(len(pts), N, replace=False)
        pts = pts[idx]
    else:
        pad = np.zeros((N - len(pts), 4), dtype=np.float32)
        pts = np.vstack([pts, pad])
    lidar_t = torch.from_numpy(pts).unsqueeze(0).to(DEVICE)   # (1, N, 4)

    with torch.no_grad(), autocast(enabled=cfg.mixed_precision):
        preds = model(img_t, lidar_t)

    # Decode detections
    logits = preds['pred_logits'][0]
    scores, classes = logits.softmax(-1)[:, :-1].max(-1)
    boxes  = box_cxcywh_to_xyxy(preds['pred_boxes'][0]).cpu().numpy()
    scores = scores.cpu().numpy()
    classes= classes.cpu().numpy()

    # Filter
    keep = scores > 0.3
    boxes, scores, classes = boxes[keep], scores[keep], classes[keep]

    # Weather
    w_probs = preds['pred_weather'][0].softmax(0).cpu().numpy()
    w_pred  = w_probs.argmax()
    cond_names = ['Clear','Fog','Rain','Snow','Night','Dust']

    return {
        'boxes'       : boxes,
        'scores'      : scores,
        'classes'     : classes,
        'seg_mask'    : preds['pred_seg'][0].argmax(0).cpu().numpy(),
        'trajectory'  : preds['pred_traj'][0].cpu().numpy(),
        'weather'     : cond_names[w_pred],
        'weather_prob': w_probs,
    }


# ── Benchmark latency ─────────────────────────────────────────
print('Benchmarking inference latency...')
sample_img = val_sample['image'][0].cpu().permute(1,2,0).numpy()
sample_img = (sample_img * IMAGENET_STD + IMAGENET_MEAN)
sample_img = (np.clip(sample_img, 0, 1) * 255).astype(np.uint8)
sample_pts = val_sample['lidar'][0].cpu().numpy()

# Warm-up
for _ in range(3):
    _ = inference_single(MODEL, sample_img, sample_pts, CFG)

# Measure
N_RUNS = 20
torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
t0 = time.perf_counter()
for _ in range(N_RUNS):
    _ = inference_single(MODEL, sample_img, sample_pts, CFG)
torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
elapsed = (time.perf_counter() - t0) / N_RUNS

print(f'\nInference latency : {elapsed*1000:.1f} ms/frame')
print(f'Throughput        : {1/elapsed:.1f} FPS')

# Run inference and display
result = inference_single(MODEL, sample_img, sample_pts, CFG)
print(f'\nDetected {len(result["boxes"])} objects')
print(f'Predicted weather: {result["weather"]}')
for i, (b, s, c) in enumerate(
        zip(result['boxes'], result['scores'], result['classes'])):
    print(f'  [{i}] {CFG.class_names[c]:12s} score={s:.3f} box={b.round(3)}')

## Cell 18 — Architecture Diagram

In [ ]:
# ============================================================
# CELL 18: Model Architecture Diagram
# Generates a publication-quality architecture figure
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(20, 10))
fig.patch.set_facecolor('#0a0a0f')
ax.set_facecolor('#0a0a0f')
ax.set_xlim(0, 20)
ax.set_ylim(0, 10)
ax.axis('off')

def draw_block(ax, x, y, w, h, label, sublabel='',
               fc='#1e3a5f', ec='#4fc3f7', text_c='white'):
    rect = mpatches.FancyBboxPatch(
        (x, y), w, h, boxstyle='round,pad=0.1',
        fc=fc, ec=ec, linewidth=1.5, zorder=3
    )
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + (0.15 if sublabel else 0),
            label, ha='center', va='center',
            color=text_c, fontsize=8.5, fontweight='bold', zorder=4)
    if sublabel:
        ax.text(x + w/2, y + h/2 - 0.25, sublabel, ha='center', va='center',
                color='#aaa', fontsize=7, zorder=4)

def draw_arrow(ax, x1, y1, x2, y2, color='#4fc3f7'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5),
                zorder=2)

# ── Inputs ───────────────────────────────────────────────────
draw_block(ax, 0.2, 6.5, 2.2, 1.5, 'RGB Camera', '(3×384×640)', fc='#1b3a1b', ec='#81c784')
draw_block(ax, 0.2, 3.5, 2.2, 1.5, 'LiDAR', '(N×4 pts)', fc='#3a1b1b', ec='#ef9a9a')

# ── Encoders ─────────────────────────────────────────────────
draw_block(ax, 3.0, 6.5, 2.5, 1.5, 'Swin Transformer', 'Image Encoder', fc='#1b2a3a', ec='#4fc3f7')
draw_block(ax, 3.0, 3.5, 2.5, 1.5, 'PointPillars', 'LiDAR Encoder', fc='#2a1b3a', ec='#ce93d8')

# ── BEV Fusion ───────────────────────────────────────────────
draw_block(ax, 6.5, 5.0, 2.5, 2.0, 'BEVFusion', 'Multi-modal Fusion\n+ Weather Gate', fc='#3a2a0a', ec='#ffd54f')

# ── Transformer ──────────────────────────────────────────────
draw_block(ax, 10.0, 5.0, 2.5, 2.0, 'DETR3D', 'Transformer Decoder\n(6L × 8H)', fc='#0a2a3a', ec='#4dd0e1')

# ── Heads ────────────────────────────────────────────────────
draw_block(ax, 13.5, 7.5, 2.5, 1.2, 'Detection Head', 'BBoxes + Classes', fc='#1e3a5f', ec='#64b5f6')
draw_block(ax, 13.5, 5.8, 2.5, 1.2, 'Seg Head', 'Semantic Masks', fc='#1b3a1b', ec='#81c784')
draw_block(ax, 13.5, 4.1, 2.5, 1.2, 'GRU Decoder', 'Trajectory', fc='#3a1b3a', ec='#f48fb1')
draw_block(ax, 13.5, 2.4, 2.5, 1.2, 'Weather Head', 'Condition Class', fc='#3a2a0a', ec='#ffcc02')

# ── Outputs ──────────────────────────────────────────────────
draw_block(ax, 17.2, 7.5, 2.5, 1.2, '3D Boxes', 'cx,cy,w,h', fc='#0d0d0d', ec='#64b5f6', text_c='#64b5f6')
draw_block(ax, 17.2, 5.8, 2.5, 1.2, 'Seg Mask', 'H×W×8cls', fc='#0d0d0d', ec='#81c784', text_c='#81c784')
draw_block(ax, 17.2, 4.1, 2.5, 1.2, 'Waypoints', 'T×2 (Δx,Δy)', fc='#0d0d0d', ec='#f48fb1', text_c='#f48fb1')
draw_block(ax, 17.2, 2.4, 2.5, 1.2, 'Weather', '6-class', fc='#0d0d0d', ec='#ffcc02', text_c='#ffcc02')

# ── Arrows ───────────────────────────────────────────────────
draw_arrow(ax, 2.4, 7.25, 3.0, 7.25, '#81c784')
draw_arrow(ax, 2.4, 4.25, 3.0, 4.25, '#ef9a9a')
draw_arrow(ax, 5.5, 7.25, 6.5, 6.5, '#4fc3f7')
draw_arrow(ax, 5.5, 4.25, 6.5, 5.5, '#ce93d8')
draw_arrow(ax, 9.0, 6.0, 10.0, 6.0, '#ffd54f')

for hy in [8.1, 6.4, 4.7, 3.0]:
    draw_arrow(ax, 12.5, 6.0, 13.5, hy)

for hy in [8.1, 6.4, 4.7, 3.0]:
    draw_arrow(ax, 16.0, hy, 17.2, hy)

# ── Augmentation box ─────────────────────────────────────────
aug_rect = mpatches.FancyBboxPatch(
    (0.1, 1.0), 5.0, 1.5, boxstyle='round,pad=0.1',
    fc='#1a0a00', ec='#ff7043', linewidth=1.5, linestyle='--', zorder=1
)
ax.add_patch(aug_rect)
ax.text(2.6, 1.75, '🌧 Weather Augmentation: Fog | Rain | Snow | Night | Dust',
        ha='center', va='center', color='#ff8a65', fontsize=8.5, zorder=2)

ax.text(10, 9.5, 'WeatherRobustDetector — Full Architecture',
        ha='center', va='center', color='white', fontsize=14,
        fontweight='bold', zorder=5)

plt.tight_layout()
arch_path = ROOT / 'outputs' / 'viz' / 'architecture_diagram.png'
plt.savefig(arch_path, dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f'Architecture diagram saved: {arch_path}')

## Cell 19 — Final Model Export & Report

In [ ]:
# ============================================================
# CELL 19: Final Export & Research Summary Report
# ============================================================

import platform, datetime

# ── Export ONNX (optional, requires onnx package) ────────────
def export_onnx(model, cfg, path):
    try:
        import onnx, onnxruntime
        model.eval()
        dummy_img   = torch.randn(1, 3, cfg.img_h, cfg.img_w).to(DEVICE)
        dummy_lidar = torch.randn(1, cfg.max_lidar_pts, 4).to(DEVICE)
        torch.onnx.export(
            model, (dummy_img, dummy_lidar),
            path,
            input_names=['image', 'lidar'],
            output_names=['pred_logits', 'pred_boxes', 'pred_seg', 'pred_traj', 'pred_weather'],
            opset_version=14,
            do_constant_folding=True,
        )
        print(f'ONNX model exported: {path}')
    except ImportError:
        print('onnx/onnxruntime not installed — skipping ONNX export.')
    except Exception as e:
        print(f'ONNX export failed: {e}')

export_onnx(MODEL, CFG, str(ROOT / 'checkpoints' / 'model.onnx'))

# ── TorchScript export ────────────────────────────────────────
# (scripting Swin backbone is complex — use trace instead)
# torch.jit.trace is skipped for multi-output dicts; use .pth

# ── Final summary report ──────────────────────────────────────
report = f"""
╔══════════════════════════════════════════════════════════════╗
║         WEATHER-ROBUST OBSTACLE DETECTION — FINAL REPORT     ║
╠══════════════════════════════════════════════════════════════╣
║ Date      : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}                             ║
║ Platform  : {platform.node()[:40]:40s}  ║
║ GPU       : {(torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU')[:40]:40s}  ║
╠══════════════════════════════════════════════════════════════╣
║ ARCHITECTURE                                                 ║
║   Backbone    : {CFG.backbone[:44]:44s} ║
║   LiDAR Enc.  : PointPillars                                 ║
║   Fusion      : BEVFusion (Channel Attention + Weather Gate) ║
║   Detector    : DETR3D Transformer ({CFG.num_transformer_layers}L × {CFG.num_heads}H, Q={CFG.num_queries})    ║
║   Seg Head    : Upsampling CNN ({CFG.num_seg_classes} classes)                  ║
║   Temporal    : 2-layer GRU ({CFG.gru_hidden} hidden)                   ║
║   Params      : {MODEL.count_params():,} trainable                 ║
╠══════════════════════════════════════════════════════════════╣
║ EVALUATION METRICS                                           ║
║   mAP              : {METRICS.get('mAP', 'N/A')}                              ║
║   Precision        : {METRICS.get('Precision', 'N/A')}                              ║
║   Recall           : {METRICS.get('Recall', 'N/A')}                              ║
║   F1 Score         : {METRICS.get('F1', 'N/A')}                              ║
║   Det. IoU         : {METRICS.get('mean_IoU_det', 'N/A')}                              ║
║   Seg. mIoU        : {METRICS.get('mIoU_seg', 'N/A')}                              ║
║   Weather Acc.     : {METRICS.get('overall_weather_acc', 'N/A')}                              ║
╠══════════════════════════════════════════════════════════════╣
║ WEATHER ROBUSTNESS                                           ║"""

for cond, acc in METRICS.get('weather_robustness', {}).items():
    report += f"\n║   {cond:8s}: {acc:.4f}                                          ║"

report += f"""
╠══════════════════════════════════════════════════════════════╣
║ OUTPUTS                                                      ║
║   Best checkpoint : checkpoints/best.pth                     ║
║   Training log    : logs/train_history.json                  ║
║   Eval metrics    : logs/eval_metrics.json                   ║
║   Visualisations  : outputs/viz/                             ║
╚══════════════════════════════════════════════════════════════╝
"""

print(report)

with open(ROOT / 'FINAL_REPORT.txt', 'w') as f:
    f.write(report)

print(f'Full report saved: {ROOT}/FINAL_REPORT.txt')

## Cell 20 — Real Waymo Dataset Integration (When Available)

In [ ]:
# ============================================================
# CELL 20: Real Waymo Dataset Integration
#
# USAGE: Uncomment and run when Waymo TFRecord files are
# downloaded to /content/waymo_data/
#
# Step 1: Register at https://waymo.com/open/
# Step 2: Download .tfrecord files via gsutil:
#   gsutil -m cp gs://waymo_open_dataset_v_1_4_1/training/* /content/waymo_data/
# Step 3: Set USE_SYNTHETIC = False in Cell 5
# Step 4: Run this cell to replace the dataset
# ============================================================

WAYMO_DATA_DIR = '/content/waymo_data'

# Uncomment below when Waymo data is available:

# import tensorflow as tf
# from waymo_open_dataset import dataset_pb2
# from waymo_open_dataset.utils import frame_utils, range_image_utils

# class WaymoRealDataset(Dataset):
#     def __init__(self, tfrecord_dir, cfg, split='train'):
#         self.cfg = cfg
#         self.tfrecords = sorted(Path(tfrecord_dir).glob('*.tfrecord'))
#         if split == 'train':
#             self.tfrecords = self.tfrecords[:int(0.8*len(self.tfrecords))]
#         else:
#             self.tfrecords = self.tfrecords[int(0.8*len(self.tfrecords)):]
#         # Build index of (file, frame_idx) pairs
#         self.index = []
#         for tf_path in self.tfrecords:
#             ds = tf.data.TFRecordDataset(str(tf_path))
#             for i, _ in enumerate(ds):
#                 self.index.append((str(tf_path), i))
#         self.transforms = build_transforms(cfg, split=='train')
#
#     def __len__(self):
#         return len(self.index)
#
#     def __getitem__(self, idx):
#         tf_path, frame_idx = self.index[idx]
#         ds = tf.data.TFRecordDataset(tf_path)
#         for i, data in enumerate(ds):
#             if i == frame_idx:
#                 break
#
#         frame = dataset_pb2.Frame()
#         frame.ParseFromString(data.numpy())
#
#         # Extract front camera image
#         for img in frame.images:
#             if img.name == dataset_pb2.CameraName.FRONT:
#                 img_np = cv2.imdecode(np.frombuffer(img.image, np.uint8), cv2.IMREAD_COLOR)
#                 img_np = cv2.cvtColor(img_np, cv2.COLOR_BGR2RGB)
#                 break
#
#         # Extract LiDAR (top lidar)
#         range_images, camera_projections, _, range_image_top_pose = (
#             frame_utils.parse_range_image_and_camera_projection(frame)
#         )
#         pts_list, _ = frame_utils.convert_range_image_to_point_cloud(
#             frame, range_images, camera_projections, range_image_top_pose
#         )
#         pts = np.concatenate([p for p in pts_list], axis=0)[:, :4]
#
#         # Extract 3D boxes → project to 2D for this demo
#         boxes, labels = [], []
#         type_map = {1:0, 2:1, 4:2}  # Vehicle, Pedestrian, Cyclist
#         for label_obj in frame.laser_labels:
#             if label_obj.type in type_map:
#                 box3d = label_obj.box
#                 # Simplified 2D projection
#                 cx, cy = box3d.center_x, box3d.center_y
#                 w, h_box = box3d.width, box3d.length
#                 # Map to image space (rough)
#                 H, W = img_np.shape[:2]
#                 x1 = max(0, int((cx + 25) / 50 * W - w * 5))
#                 y1 = max(0, int((25 - cy) / 50 * H - h_box * 3))
#                 x2 = min(W-1, x1 + int(w * 10))
#                 y2 = min(H-1, y1 + int(h_box * 6))
#                 if x2 > x1 and y2 > y1:
#                     boxes.append([x1, y1, x2, y2])
#                     labels.append(type_map[label_obj.type])
#
#         # Apply transforms, weather aug, etc. (same as synthetic)
#         # ... (identical to WaymoSyntheticDataset.__getitem__)
#         pass  # Fill in as above
#
# # To use:
# # TRAIN_DS = WaymoRealDataset(WAYMO_DATA_DIR, CFG, 'train')
# # TRAIN_LOADER = DataLoader(TRAIN_DS, ...)

print('Real Waymo dataset integration template ready.')
print('Follow the instructions in Cell 20 comments to use real data.')